In [1]:
SMOKE = False
JOBS = ['0-5', '6-11']
import glob, os, subprocess
t = glob.glob('/kaggle/input/**/leakbench/train.py', recursive=True)[0]
ROOT = t.rsplit('/leakbench/', 1)[0]
print('ROOT', ROOT)
subprocess.run('pip -q install segmentation-models-pytorch==0.5.0 timm', shell=True)
subprocess.run(f'rm -rf /kaggle/working/leakbench && cp -r {ROOT}/leakbench /kaggle/working/leakbench', shell=True)
W = '/kaggle/working/leakbench'
if SMOKE: cmds = [f'CUDA_VISIBLE_DEVICES={g} python {W}/train.py --data-root {ROOT} --manifest {W}/manifests/v2.csv --model {m} --epochs 1 --limit 32 --out /kaggle/working/smoke' for g, m in ((0, 'unet_r34'), (1, 'xattn'))]
else: cmds = [f'CUDA_VISIBLE_DEVICES={g} python {W}/run_queue.py --data-root {ROOT} --jobs {j} --out /kaggle/working/runs --budget-hours 11.5 > /kaggle/working/log_gpu{g}.txt 2>&1' for g, j in enumerate(JOBS)]
procs = [subprocess.Popen(c, shell=True) for c in cmds]
print([p.wait() for p in procs])
subprocess.run(f'python {W}/aggregate.py ' + ('/kaggle/working/smoke' if SMOKE else '/kaggle/working/runs'), shell=True)

ROOT /kaggle/input/datasets/gokulrocky/chakra-leakbench-v1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.5 MB/s eta 0:00:00
[0, 0]
v2.csv     fpn_pvtv2b2    n=3 val=0.898±0.004 kvasir=0.917±0.006 clinic=0.909±0.015 colon=0.771±0.006 etis=0.776±0.009 cvc300=0.880±0.014 FA=0.843±0.133
v2.csv     segformer_b2   n=3 val=0.901±0.008 kvasir=0.918±0.001 clinic=0.897±0.006 colon=0.773±0.003 etis=0.771±0.015 cvc300=0.859±0.011 FA=0.838±0.037
v2.csv     unet_r34       n=3 val=0.884±0.008 kvasir=0.892±0.017 clinic=0.899±0.015 colon=0.701±0.015 etis=0.659±0.037 cvc300=0.859±0.014 FA=0.217±0.032
v2.csv     xattn          n=3 val=0.891±0.009 kvasir=0.908±0.011 clinic=0.898±0.018 colon=0.750±0.005 etis=0.746±0.023 cvc300=0.865±0.000 FA=0.562±0.095


CompletedProcess(args='python /kaggle/working/leakbench/aggregate.py /kaggle/working/runs', returncode=0)